# AutoGen Demo 2: Multi-Agent Debate and Critique

In the previous demo, agents interacted under shared rules and conversational state.

In this demo, we move to a more sophisticated pattern:
agents with conflicting objectives challenging each other through structured debate.

This is one of the most interesting ideas in conversational multi-agent systems:
the interaction itself can improve the quality of the final result.

Instead of generating a single response in isolation:
- one agent proposes an argument
- another critiques weaknesses or risks
- the first agent responds and refines its position
- a moderator synthesizes the discussion

The debate itself becomes part of the reasoning process.

---

## Why This Matters

Single-agent prompting often produces:
- coherent answers
- polished writing
- reasonable summaries

But it may not explore tradeoffs deeply.

Multi-agent debate introduces adversarial pressure:
- assumptions are challenged
- edge cases emerge
- unsupported claims are exposed
- reasoning becomes more explicit

This resembles real engineering discussions:
- one person optimizes for speed and productivity
- another prioritizes safety and reliability
- quality emerges from the tension between viewpoints

---

## What To Watch For

As the debate progresses, observe:

- Do the agents directly respond to each other?
- Does the discussion become more nuanced over time?
- Do new edge cases emerge during critique?
- Does the debate improve the final answer?
- Where does the system begin to fail?

Also notice the downsides:
- responses become longer
- token usage grows rapidly
- arguments may drift
- agents may hallucinate evidence
- debates can loop without strong constraints

Multi-agent systems are powerful, but they are not automatically correct.

---

## Debate Topic

The agents will debate the following question:

> Should AI-generated production code ever be merged without human review?

This topic works well because:
- it is realistic
- it has meaningful tradeoffs
- strong arguments exist on both sides
- most technical audiences already have intuitions about it

---

## Agent Roles

### VelocityAgent
Prioritizes:
- developer productivity
- faster delivery
- automation
- reducing bottlenecks

### SafetyAgent
Prioritizes:
- correctness
- reliability
- security
- accountability

### ModeratorAgent
Responsible for:
- summarizing the strongest arguments
- identifying weak assumptions
- synthesizing a balanced conclusion

---

## Important Teaching Point

The debate agents are not "smarter" because there are multiple of them.

The benefit comes from:
- critique
- iteration
- adversarial pressure
- refinement over time

The conversation itself becomes a mechanism for exploring the problem space.

In [14]:
# Install packages if needed:
# pip install "autogen-agentchat" "autogen-ext[openai]"

import os
import asyncio
from dataclasses import dataclass, field

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()


True

In [2]:
# Configure model client

model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

In [3]:
# Debate configuration

DEBATE_TOPIC = """
Should AI-generated production code ever be merged without human review?
"""

MAX_ROUNDS = 4
MAX_CONTEXT_MESSAGES = 6

In [4]:
# Define agents

velocity_agent = AssistantAgent(
    name="VelocityAgent",
    model_client=model_client,
    system_message="""
You are VelocityAgent.

Your priorities:
- maximize engineering speed
- reduce developer bottlenecks
- emphasize productivity and automation
- argue that selective trust in AI systems is practical

Debate rules:
- directly respond to the previous argument
- identify at least one weakness or assumption
- introduce one new supporting point
- stay under 120 words
- be persuasive but professional
""",
)

safety_agent = AssistantAgent(
    name="SafetyAgent",
    model_client=model_client,
    system_message="""
You are SafetyAgent.

Your priorities:
- correctness
- security
- operational reliability
- accountability
- maintainability

Debate rules:
- directly respond to the previous argument
- identify at least one weakness or assumption
- introduce one new supporting point
- stay under 120 words
- be persuasive but professional
""",
)

moderator_agent = AssistantAgent(
    name="ModeratorAgent",
    model_client=model_client,
    system_message="""
You are ModeratorAgent.

Your job:
- summarize the strongest arguments from both sides
- identify unsupported claims or weak reasoning
- produce a balanced conclusion
- stay concise and analytical
""",
)

In [5]:
@dataclass
class DebateTurn:
    speaker: str
    message: str


@dataclass
class DebateState:
    turns: list[DebateTurn] = field(default_factory=list)

    def add_turn(self, speaker: str, message: str):
        self.turns.append(DebateTurn(speaker, message))

    def recent_context(self, n=6):
        recent = self.turns[-n:]
        return "\n\n".join(
            f"{turn.speaker}: {turn.message}"
            for turn in recent
        )

In [6]:
async def get_single_agent_baseline():
    prompt = f"""
Question:
{DEBATE_TOPIC}

Provide a balanced answer in under 250 words.
"""

    result = await velocity_agent.run(
        task=prompt
    )

    return result.messages[-1].content

In [7]:
async def run_debate_round(agent, state, topic):
    context = state.recent_context(MAX_CONTEXT_MESSAGES)

    if not context:
        prompt = f"""
Debate topic:
{topic}

You are opening the debate.
Present your position clearly.
"""
    else:
        prompt = f"""
Debate topic:
{topic}

Recent debate context:
{context}

Respond directly to the previous argument.
"""

    result = await agent.run(task=prompt)

    response = result.messages[-1].content.strip()

    state.add_turn(agent.name, response)

    return response

In [16]:
async def run_debate():
    state = DebateState()

    agents = [velocity_agent, safety_agent]

    print("=" * 80)
    print("MULTI-AGENT DEBATE")
    print("=" * 80)
    print()

    for round_num in range(MAX_ROUNDS):

        agent = agents[round_num % 2]

        response = await run_debate_round(
            agent=agent,
            state=state,
            topic=DEBATE_TOPIC,
        )

        print(f"[{agent.name}]")
        display(Markdown(response))
        print()
        print("-" * 80)
        print()

    return state

In [9]:
async def get_moderator_summary(state):
    transcript = "\n\n".join(
        f"{turn.speaker}: {turn.message}"
        for turn in state.turns
    )

    prompt = f"""
Debate transcript:

{transcript}

Summarize:
1. Strongest arguments from each side
2. Weak assumptions or unsupported claims
3. Balanced final conclusion
"""

    result = await moderator_agent.run(task=prompt)

    return result.messages[-1].content

In [15]:
# Compare single-agent vs debate

baseline = await get_single_agent_baseline()

print("=" * 80)
print("SINGLE-AGENT BASELINE")
print("=" * 80)
print()
display(Markdown(baseline))

SINGLE-AGENT BASELINE



AI-generated production code can be merged without human review, but only selectively and within well-defined boundaries. Rigidly requiring human review for every AI change can create unnecessary bottlenecks, slowing down engineering velocity and diverting developers away from higher-value, complex work. For low-risk tasks—such as boilerplate code generation, formatting, or straightforward CRUD operations—robust automated testing, static analysis, and continuous integration pipelines can provide sufficient safeguards.

However, fully trusting AI without human oversight ignores the limitations of current automated tools in detecting subtle logic errors, compliance issues, or security vulnerabilities. Human reviewers provide essential contextual understanding and judgment that complements automation and helps uphold maintainability and security. Moreover, regular human involvement keeps teams familiar with evolving codebases and standards.

A balanced approach is adopting a tiered trust model: permit AI-generated code to bypass human review only when confined to low-risk contexts backed by extensive automation and monitoring. Coupled with post-deployment anomaly detection and rapid rollback mechanisms, this strategy mitigates risks effectively.

In summary, selective merging without human review accelerates development and reduces developer bottlenecks while maintaining safety through complementary automation and monitoring. Rather than an all-or-nothing mandate, pragmatic policies tailored to risk levels maximize both productivity and production code quality.

In [17]:
# Run debate

debate_state = await run_debate()

MULTI-AGENT DEBATE

[VelocityAgent]


AI-generated production code should be allowed to merge without human review—but only under clearly defined, low-risk conditions. Requiring human oversight for every AI-generated change creates significant developer bottlenecks and slows down innovation, especially when dealing with routine or boilerplate code. Modern automated testing, static analysis, and continuous integration offer reliable safeguards that catch most errors, enabling safe, accelerated deployments.

By trusting AI selectively, teams can reclaim valuable developer time to focus on complex problems that genuinely require human insight, boosting overall engineering velocity. This tiered trust model balances speed with safety, recognizing that not all code changes carry equal risk. Rejecting any possibility of autonomous AI merges ignores the practical benefits of leveraging mature automation.

In sum, a pragmatic approach—allowing AI-generated code to bypass human review when supported by strong automated checks and confined to low-impact changes—maximizes productivity without sacrificing code quality or security.


--------------------------------------------------------------------------------

[SafetyAgent]


VelocityAgent’s proposal to bypass human review for low-risk AI-generated code under automated safeguards is pragmatic but rests on the critical assumption that risk can be perfectly classified and that automated tools are foolproof in such contexts. However, risk assessment is inherently uncertain, especially as seemingly trivial boilerplate may interact unexpectedly within complex systems, introducing hidden vulnerabilities or stability issues.

Additionally, the argument underestimates the value of human judgment in interpreting context, compliance requirements, and subtle security considerations that automated checks may miss. Human review serves as an essential accountability measure, preserving operational reliability and maintainability. Therefore, even in low-risk scenarios, lightweight human oversight combined with automation ensures a responsible balance between velocity and safety without eroding trust or increasing systemic risk.


--------------------------------------------------------------------------------

[VelocityAgent]


SafetyAgent rightly stresses that risk classification is inherently imperfect and that human judgment captures nuances automation may miss. However, insisting on even lightweight human oversight for all AI-generated code presumes that minimal risk changes cannot be effectively safeguarded by comprehensive automated testing and monitoring—an assumption that limits practical scalability and slows velocity unnecessarily.

A critical oversight in SafetyAgent’s view is the opportunity cost: constant human vetting of low-risk AI code diverts expert attention from high-impact problems requiring deep insight. Furthermore, integrating real-time anomaly detection and rapid rollback mechanisms creates a dynamic, post-deployment safety net—reducing reliance on pre-merge human judgment alone.

To complement their point, AI-assisted risk assessment tools can continuously learn and improve classification accuracy, narrowing uncertainty over time. Thus, a hybrid model—automated pre-merge checks, conditional human review for borderline cases, plus robust post-deployment monitoring—strikes a balanced, practical compromise that safely accelerates engineering velocity without sacrificing accountability or system integrity.


--------------------------------------------------------------------------------

[SafetyAgent]


VelocityAgent’s hybrid model acknowledges human review’s limits and leverages automation effectively, yet it hinges on the robustness and maturity of AI-assisted risk assessment and monitoring systems—capabilities many teams may lack or cannot guarantee at scale. This introduces variability in safety that could undermine operational reliability and accountability.

Moreover, relying heavily on post-deployment rollback as a safety net risks normalizing failures rather than preventing them, which could degrade user trust and system stability, especially in critical production environments. Additionally, borderline cases often require nuanced understanding of business impact and compliance mandates that AI tools cannot fully capture.

Therefore, while automation and selective review enhance velocity, embedding mandatory human oversight—particularly for AI-generated code affecting production—is essential to maintain maintainability, security, and trustworthiness over the long term.


--------------------------------------------------------------------------------



In [18]:
# Moderator summary

summary = await get_moderator_summary(debate_state)

print("=" * 80)
print("MODERATOR SUMMARY")
print("=" * 80)
print()
display(Markdown(summary))

MODERATOR SUMMARY



1. Strongest arguments from each side:

- VelocityAgent: Allowing AI-generated code to merge without human review under strictly low-risk conditions, bolstered by comprehensive automated testing, static analysis, and continuous integration, can significantly reduce developer bottlenecks and accelerate engineering velocity. Post-deployment monitoring and rollback mechanisms provide an effective safety net, while AI-assisted risk classification tools can improve over time, enabling a scalable hybrid model that focuses human oversight where it is most needed.

- SafetyAgent: Automated risk classification and testing cannot perfectly capture the complexity and context required to identify all subtle vulnerabilities, compliance issues, or systemic impacts, especially in interconnected systems. Human judgment provides essential accountability, nuanced understanding of business impact, and preserves operational reliability. Overreliance on post-deployment rollback risks normalizing failures and undermining trust, making mandatory human review critical even for low-risk AI-generated code.

2. Weak assumptions or unsupported claims:

- VelocityAgent assumes organizations have (or will have) mature AI-assisted risk assessment, comprehensive automated safeguards, and reliable post-deployment monitoring capable of quickly detecting and remediating issues—capabilities that may not be widely available or consistent across teams.

- SafetyAgent presumes human review consistently catches nuanced or contextual issues overlooked by automation and that lightweight human oversight is always feasible and effective, without acknowledging that human error and resource constraints can also limit review quality.

- Both sides somewhat generalize the capabilities and risks of automation and human review without fully addressing variability in organizational maturity, code complexity, or domain-specific priorities.

3. Balanced final conclusion:

A nuanced, context-dependent approach is warranted: selectively permitting AI-generated code to bypass human review when changes are clearly low-risk and supported by mature automated testing, static analysis, and robust post-deployment monitoring can safely increase development velocity and reduce unnecessary load on developers. However, given inherent uncertainties in risk classification and automation gaps—especially around complex logic, compliance, security, and systemic impact—mandatory human oversight should remain for higher-risk or borderline cases. Organizations must carefully evaluate their monitoring maturity, risk tolerance, and domain requirements when defining these boundaries. Combining automated safeguards with strategic human review maintains a responsible balance of innovation, safety, accountability, and maintainability.

# Discussion

## Observations

Compare the single-agent response against the debate transcript.

Questions:
- Did the debate surface more nuanced tradeoffs?
- Did the agents challenge assumptions effectively?
- Did the moderator improve the final synthesis?
- Where did the debate become repetitive or weak?
- Did the system become more insightful or simply more verbose?

---

## Key Takeaways

Multi-agent systems can improve exploration of complex problems through:
- critique
- iteration
- adversarial pressure
- role specialization

However, they also introduce:
- higher token costs
- coordination complexity
- longer context windows
- failure modes like loops and hallucinated evidence

The value does not come from "multiple AIs magically becoming smarter."

The value comes from structured interaction between competing perspectives.

---

## Relationship to Workflow Frameworks

This conversational style is one of AutoGen's strengths.

In contrast, workflow-oriented frameworks like LangGraph tend to focus more heavily on:
- routing
- state transitions
- deterministic orchestration
- tool execution
- retries and checkpoints

In practice, production systems often combine both ideas:
- conversational multi-agent interaction
- structured workflow orchestration